# Домашнее задание 2. Компьютерный практикум 3.

Печенин Данила Михайлович. БПМ233.

In [1]:
import numpy as np
from numba import njit, prange, objmode
import matplotlib.pyplot as plt
import time

In [5]:
@njit(parallel=True)
def compute_normed_mean_energy(Lx_array: np.ndarray, Ly: int, kT_array: np.ndarray, J: float = 1.0) -> np.ndarray:
    Lx_array_len, kT_array_len = Lx_array.shape[0], kT_array.shape[0]
    normed_mean_energies = np.zeros((Lx_array_len, kT_array_len))
    for Lx_idx in range(Lx_array_len):
        with objmode(start='f8'):
            start = time.time()
        
        for kT_idx in range(kT_array_len):
            Lx = Lx_array[Lx_idx]
            kT = kT_array[kT_idx]
            area = Lx * Ly
            states_len = 2**area

            energies = np.empty(states_len)
            probabilities = np.empty(states_len)

            for state in prange(states_len):
                E = 0.0
                spins = np.empty(area, dtype=np.int8)
                state_flag = np.int64(state)
                for i in range(area):
                    spins[i] = 1 if (state_flag & 1) == 1 else -1
                    state_flag >>= 1
                spins = spins.reshape((Lx, Ly))

                for i in range(Lx):
                    for j in range(Ly):
                        E -= J * spins[i, j] * (spins[(i + 1) % Lx, j] + spins[i, (j + 1) % Ly])

                probability = np.exp(-E / kT)
                energies[state] = E
                probabilities[state] = probability
            
            Z = np.sum(probabilities)
            normed_mean_energies[Lx_idx, kT_idx] = np.sum(energies * probabilities) / Z / area

        with objmode():
            print("Lx =", Lx, "done in", time.time() - start, "seconds")

    return normed_mean_energies

In [7]:
kT_array = np.arange(1.0, 10.0, 0.2)

In [8]:
compute_normed_mean_energy(np.array([15]), 2, kT_array)

Lx = 15 done in 272.20036029815674 seconds


array([[-1.98718948, -1.95115673, -1.88016543, -1.78175297, -1.67149821,
        -1.5598542 , -1.45173992, -1.34971935, -1.25527306, -1.16905924,
        -1.09108778, -1.02094056, -0.95797131, -0.90144821, -0.85064053,
        -0.80486456, -0.76350358, -0.72601326, -0.69191908, -0.66081029,
        -0.63233254, -0.60618046, -0.58209072, -0.5598359 , -0.53921915,
        -0.52006958, -0.50223843, -0.48559579, -0.47002779, -0.45543439,
        -0.4417273 , -0.42882847, -0.41666858, -0.40518597, -0.39432555,
        -0.38403803, -0.37427916, -0.36500909, -0.35619187, -0.34779499,
        -0.33978896, -0.33214698, -0.32484466, -0.31785973, -0.31117183]])